# LSEG 美债收益率 & 美元晨间看板（v6 深圳早会洁净版）

**定位**：每天深圳时间 9:00 左右，复盘上一中国交易日收盘后至今早的海外市场变化。

这版只保留早会真正有用的模块，并按阅读顺序排列：

1. **美债名义收益率**：2Y / 5Y / 10Y / 30Y  
2. **曲线结构**：2s10s / 5s10s / 5s30s  
3. **真实利率与通胀预期**：TIPS real yield / BEI = Nominal - TIPS  
4. **美债期货**：TU / FV / TY / US / WN  
5. **美元与外汇**：DXY / EURUSD / USDJPY / GBPUSD / AUDUSD  
6. **人民币**：USD/CNY 中间价、USDCNY、USDCNH、CNH-CNY spread、spot-fixing gap  
7. **商品**：Brent / WTI / Gold / Copper  
8. **日历与交易时段**：美国重要数据时间、主要品种交易时段、数据质量检查  

默认主窗口：**上一中国交易日 16:00 → 今日 09:00（Asia/Shanghai）**。  
辅助窗口：**上一纽约交易日 08:00 → 17:00 ET**、**截至 09:00 的滚动 24 小时**。

In [ ]:
# === 0. 安装依赖 ===
!pip -q install lseg-data pandas numpy plotly pandas-market-calendars

In [ ]:
# === 1. 基础设置 ===
from __future__ import annotations

import os
import math
import warnings
from getpass import getpass
from datetime import datetime, timedelta, time as dt_time
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML, Markdown

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 200
pd.options.display.width = 200

REPORT_TZ = ZoneInfo("Asia/Shanghai")
NY_TZ = ZoneInfo("America/New_York")

# 早会口径
REPORT_AS_OF_HOUR = 9
REPORT_AS_OF_MINUTE = 0
CN_MARKET_CLOSE_HOUR = 16
INTRADAY_LOOKBACK_HOURS = 72
INTRADAY_INTERVAL = "5min"
LONG_LOOKBACK_DAYS = 400
LONG_INTERVAL = "daily"

# 人民币中间价通常在中国交易日上午 09:15 左右发布。
RMB_FIXING_RELEASE_HOUR = 9
RMB_FIXING_RELEASE_MINUTE = 15

OUTPUT_DIR = "/content" if os.path.exists("/content") else "."
plotly_template = "plotly_white"

# 可手动覆盖节假日/特殊工作日，格式：YYYY-MM-DD
CN_HOLIDAY_OVERRIDES = []
CN_EXTRA_WORKDAY_OVERRIDES = []
US_HOLIDAY_OVERRIDES = []
US_EXTRA_WORKDAY_OVERRIDES = []

In [ ]:
# === 2. 凭据与 LSEG Session ===

def get_secret(name: str, prompt: str, hide: bool = True) -> str:
    # 优先读取环境变量 / Colab Secrets；没有则运行时输入。
    value = os.getenv(name)
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass(prompt) if hide else input(prompt)
    if not value:
        raise ValueError(f"Missing credential: {name}")
    return value

APP_KEY = get_secret("LSEG_APP_KEY", "LSEG App Key: ", hide=True)
LDP_LOGIN = get_secret("LSEG_LDP_LOGIN", "LSEG LDP Login / Machine ID: ", hide=False)
LDP_PASSWORD = get_secret("LSEG_LDP_PASSWORD", "LSEG LDP Password: ", hide=True)

import lseg.data as ld

def open_lseg_platform_session(app_key: str, username: str, password: str):
    # Open LSEG Data Platform session using password grant.
    try:
        definition = ld.session.platform.Definition(
            app_key=app_key,
            grant=ld.session.platform.GrantPassword(username=username, password=password),
            signon_control=True,
        )
    except TypeError:
        definition = ld.session.platform.Definition(
            app_key=app_key,
            grant=ld.session.platform.GrantPassword(username=username, password=password),
        )

    session = definition.get_session()
    session.open()

    try:
        ld.session.set_default(session)
    except Exception:
        pass

    return session

session = open_lseg_platform_session(APP_KEY, LDP_LOGIN, LDP_PASSWORD)
print("LSEG session opened.")

In [ ]:
# === 3. 日期与窗口处理 ===

try:
    import pandas_market_calendars as mcal
except Exception:
    mcal = None

def _date_str(d) -> str:
    return pd.Timestamp(d).strftime("%Y-%m-%d")

def _override_date_set(xs):
    return set(pd.Timestamp(x).date() for x in xs)

CN_HOLIDAYS = _override_date_set(CN_HOLIDAY_OVERRIDES)
CN_EXTRA_WORKDAYS = _override_date_set(CN_EXTRA_WORKDAY_OVERRIDES)
US_HOLIDAYS = _override_date_set(US_HOLIDAY_OVERRIDES)
US_EXTRA_WORKDAYS = _override_date_set(US_EXTRA_WORKDAY_OVERRIDES)

def get_exchange_days(calendar_name: str, start_date, end_date) -> set:
    start_date = pd.Timestamp(start_date).date()
    end_date = pd.Timestamp(end_date).date()
    if mcal is not None:
        try:
            cal = mcal.get_calendar(calendar_name)
            sched = cal.schedule(start_date=_date_str(start_date), end_date=_date_str(end_date))
            return set(pd.Timestamp(x).date() for x in sched.index)
        except Exception:
            pass
    return set(pd.date_range(start_date, end_date, freq="B").date)

def is_cn_market_day(d) -> bool:
    d = pd.Timestamp(d).date()
    if d in CN_EXTRA_WORKDAYS:
        return True
    if d in CN_HOLIDAYS:
        return False
    return d in get_exchange_days("XSHG", d - timedelta(days=7), d + timedelta(days=7))

def is_us_market_day(d) -> bool:
    d = pd.Timestamp(d).date()
    if d in US_EXTRA_WORKDAYS:
        return True
    if d in US_HOLIDAYS:
        return False
    return d in get_exchange_days("XNYS", d - timedelta(days=7), d + timedelta(days=7))

def prev_market_day(d, market: str = "CN", inclusive: bool = False):
    d = pd.Timestamp(d).date()
    check = is_cn_market_day if market.upper() == "CN" else is_us_market_day
    cur = d if inclusive else d - timedelta(days=1)
    for _ in range(30):
        if check(cur):
            return cur
        cur -= timedelta(days=1)
    raise RuntimeError(f"Cannot find previous {market} market day near {d}")

def latest_report_asof(now: datetime | None = None) -> datetime:
    # 锚定最近一个深圳 09:00，避免下午调试时窗口漂移。
    now = now or datetime.now(REPORT_TZ)
    if now.tzinfo is None:
        now = now.replace(tzinfo=REPORT_TZ)
    now = now.astimezone(REPORT_TZ)
    anchor = now.replace(hour=REPORT_AS_OF_HOUR, minute=REPORT_AS_OF_MINUTE, second=0, microsecond=0)
    if now < anchor:
        anchor -= timedelta(days=1)
    return anchor

asof_dt = latest_report_asof()
asof_date = asof_dt.date()

prev_cn_day = prev_market_day(asof_date, "CN", inclusive=False)
main_start = datetime.combine(prev_cn_day, dt_time(CN_MARKET_CLOSE_HOUR, 0), tzinfo=REPORT_TZ)
main_end = asof_dt

rolling24_start = asof_dt - timedelta(hours=24)
rolling24_end = asof_dt

# 上一纽约交易日：取 as-of 之前最近的美国交易日，窗口设为 08:00-17:00 ET。
asof_ny_date = asof_dt.astimezone(NY_TZ).date()
prev_us_day = prev_market_day(asof_ny_date, "US", inclusive=True)
ny_close_candidate = datetime.combine(prev_us_day, dt_time(17, 0), tzinfo=NY_TZ)
if ny_close_candidate > asof_dt.astimezone(NY_TZ):
    prev_us_day = prev_market_day(prev_us_day, "US", inclusive=False)

ny_start = datetime.combine(prev_us_day, dt_time(8, 0), tzinfo=NY_TZ).astimezone(REPORT_TZ)
ny_end = datetime.combine(prev_us_day, dt_time(17, 0), tzinfo=NY_TZ).astimezone(REPORT_TZ)

intraday_start = asof_dt - timedelta(hours=INTRADAY_LOOKBACK_HOURS)
intraday_end = asof_dt
daily_start = asof_dt - timedelta(days=LONG_LOOKBACK_DAYS)
daily_end = asof_dt

# 中间价目标日期：9:15 前，今日 fixing 大概率未发布，因此目标为上一中国交易日。
fixing_release_dt = asof_dt.replace(
    hour=RMB_FIXING_RELEASE_HOUR,
    minute=RMB_FIXING_RELEASE_MINUTE,
    second=0,
    microsecond=0,
)
if asof_dt >= fixing_release_dt and is_cn_market_day(asof_date):
    target_fixing_date = asof_date
else:
    target_fixing_date = prev_market_day(asof_date, "CN", inclusive=False)

print("As-of:", asof_dt.strftime("%Y-%m-%d %H:%M %Z"))
print("主窗口:", main_start.strftime("%Y-%m-%d %H:%M"), "→", main_end.strftime("%Y-%m-%d %H:%M"))
print("纽约窗口:", ny_start.strftime("%Y-%m-%d %H:%M"), "→", ny_end.strftime("%Y-%m-%d %H:%M"), "(Asia/Shanghai)")
print("Fixing target date:", target_fixing_date)

In [ ]:
# === 4. 指标池：只保留早会核心资产，且顺序即报告顺序 ===

# 注意：BEI 不再拉 direct RIC，直接用 Nominal UST yield - TIPS real yield 计算。

ASSETS = [
    # A. 美债名义收益率
    {"key": "ust2y",  "section": "A. Rates", "group": "Nominal UST", "name": "UST 2Y",  "rics": ["US2YT=RR", "US2YT=X"],   "unit": "yield_pct", "freq": "both", "order": 10},
    {"key": "ust5y",  "section": "A. Rates", "group": "Nominal UST", "name": "UST 5Y",  "rics": ["US5YT=RR", "US5YT=X"],   "unit": "yield_pct", "freq": "both", "order": 20},
    {"key": "ust10y", "section": "A. Rates", "group": "Nominal UST", "name": "UST 10Y", "rics": ["US10YT=RR", "US10YT=X"], "unit": "yield_pct", "freq": "both", "order": 30},
    {"key": "ust30y", "section": "A. Rates", "group": "Nominal UST", "name": "UST 30Y", "rics": ["US30YT=RR", "US30YT=X"], "unit": "yield_pct", "freq": "both", "order": 40},

    # B. TIPS真实利率
    {"key": "tips5y",  "section": "B. Real & Inflation", "group": "Real Yield / TIPS", "name": "TIPS real 5Y",  "rics": ["US5YTIP=RR", "US5YTIPS=RR"],   "unit": "yield_pct", "freq": "both", "order": 110},
    {"key": "tips10y", "section": "B. Real & Inflation", "group": "Real Yield / TIPS", "name": "TIPS real 10Y", "rics": ["US10YTIP=RR", "US10YTIPS=RR"], "unit": "yield_pct", "freq": "both", "order": 120},
    {"key": "tips30y", "section": "B. Real & Inflation", "group": "Real Yield / TIPS", "name": "TIPS real 30Y", "rics": ["US30YTIP=RR", "US30YTIPS=RR"], "unit": "yield_pct", "freq": "both", "order": 130},

    # C. 美债期货
    {"key": "tu", "section": "C. Treasury Futures", "group": "Treasury Futures", "name": "TU 2Y Treasury Fut",    "rics": ["TUc1"], "unit": "futures_price", "freq": "both", "order": 210},
    {"key": "fv", "section": "C. Treasury Futures", "group": "Treasury Futures", "name": "FV 5Y Treasury Fut",    "rics": ["FVc1"], "unit": "futures_price", "freq": "both", "order": 220},
    {"key": "ty", "section": "C. Treasury Futures", "group": "Treasury Futures", "name": "TY 10Y Treasury Fut",   "rics": ["TYc1"], "unit": "futures_price", "freq": "both", "order": 230},
    {"key": "us", "section": "C. Treasury Futures", "group": "Treasury Futures", "name": "US 30Y Treasury Fut",   "rics": ["USc1"], "unit": "futures_price", "freq": "both", "order": 240},
    {"key": "wn", "section": "C. Treasury Futures", "group": "Treasury Futures", "name": "WN Ultra Bond Fut",     "rics": ["WNc1"], "unit": "futures_price", "freq": "both", "order": 250},

    # D. 美元与G10
    {"key": "dxy",    "section": "D. USD & FX", "group": "USD", "name": "DXY",    "rics": [".DXY"], "unit": "index", "freq": "both", "order": 310},
    {"key": "eurusd", "section": "D. USD & FX", "group": "FX",  "name": "EURUSD", "rics": ["EUR="], "unit": "fx",    "freq": "both", "order": 320},
    {"key": "usdjpy", "section": "D. USD & FX", "group": "FX",  "name": "USDJPY", "rics": ["JPY="], "unit": "fx",    "freq": "both", "order": 330},
    {"key": "gbpusd", "section": "D. USD & FX", "group": "FX",  "name": "GBPUSD", "rics": ["GBP="], "unit": "fx",    "freq": "both", "order": 340},
    {"key": "audusd", "section": "D. USD & FX", "group": "FX",  "name": "AUDUSD", "rics": ["AUD="], "unit": "fx",    "freq": "both", "order": 350},

    # E. 人民币
    {"key": "usdcny", "section": "E. RMB", "group": "RMB", "name": "USDCNY", "rics": ["CNY="], "unit": "fx", "freq": "both", "order": 410},
    {"key": "usdcnh", "section": "E. RMB", "group": "RMB", "name": "USDCNH", "rics": ["CNH="], "unit": "fx", "freq": "both", "order": 420},
    {"key": "cnyfix", "section": "E. RMB", "group": "RMB", "name": "USD/CNY fixing", "rics": ["CNY=PBOC", "CNYPBOC=CFXS", "CNYFIX=CFXS", "USDCNYFIX=CFXS", "CNY=SAEC"], "unit": "fixing_fx", "freq": "daily", "order": 430},

    # F. 商品
    {"key": "brent",  "section": "F. Commodities", "group": "Commodities", "name": "Brent crude", "rics": ["LCOc1"], "unit": "commodity", "freq": "both", "order": 510},
    {"key": "wti",    "section": "F. Commodities", "group": "Commodities", "name": "WTI crude",   "rics": ["CLc1"],  "unit": "commodity", "freq": "both", "order": 520},
    {"key": "gold",   "section": "F. Commodities", "group": "Commodities", "name": "Gold",        "rics": ["GCc1"],  "unit": "commodity", "freq": "both", "order": 530},
    {"key": "copper", "section": "F. Commodities", "group": "Commodities", "name": "Copper",      "rics": ["HGc1"],  "unit": "commodity", "freq": "both", "order": 540},
]

BASE_META = pd.DataFrame(ASSETS)
ORDER_BY_NAME = dict(zip(BASE_META["name"], BASE_META["order"]))
UNIT_BY_NAME = dict(zip(BASE_META["name"], BASE_META["unit"]))
GROUP_BY_NAME = dict(zip(BASE_META["name"], BASE_META["group"]))
SECTION_BY_NAME = dict(zip(BASE_META["name"], BASE_META["section"]))

DERIVED_META_ROWS = [
    {"name": "UST 2s10s spread", "section": "A. Rates", "group": "Curve", "unit": "spread_bp", "order": 60},
    {"name": "UST 5s10s spread", "section": "A. Rates", "group": "Curve", "unit": "spread_bp", "order": 70},
    {"name": "UST 5s30s spread", "section": "A. Rates", "group": "Curve", "unit": "spread_bp", "order": 80},
    {"name": "BEI 5Y",  "section": "B. Real & Inflation", "group": "Breakeven Inflation", "unit": "bei_pct", "order": 150},
    {"name": "BEI 10Y", "section": "B. Real & Inflation", "group": "Breakeven Inflation", "unit": "bei_pct", "order": 160},
    {"name": "BEI 30Y", "section": "B. Real & Inflation", "group": "Breakeven Inflation", "unit": "bei_pct", "order": 170},
    {"name": "CNH-CNY spread", "section": "E. RMB", "group": "RMB", "unit": "fx_spread", "order": 440},
    {"name": "USDCNY - fixing", "section": "E. RMB", "group": "RMB", "unit": "fx_spread", "order": 450},
    {"name": "USDCNH - fixing", "section": "E. RMB", "group": "RMB", "unit": "fx_spread", "order": 460},
]
DERIVED_META = pd.DataFrame(DERIVED_META_ROWS)
for _, r in DERIVED_META.iterrows():
    ORDER_BY_NAME[r["name"]] = r["order"]
    UNIT_BY_NAME[r["name"]] = r["unit"]
    GROUP_BY_NAME[r["name"]] = r["group"]
    SECTION_BY_NAME[r["name"]] = r["section"]

def wanted_assets(freq: str):
    if freq == "intraday":
        return [x for x in ASSETS if x["freq"] in ("both", "intraday")]
    if freq == "daily":
        return [x for x in ASSETS if x["freq"] in ("both", "daily")]
    return ASSETS

In [ ]:
# === 5. 数据拉取函数 ===

FIELD_SETS_BY_UNIT = {
    "yield_pct": [
        ["MID_YLD_1"],
        ["YLDTOMAT"],
        ["B_YLD_1", "A_YLD_1"],
        ["B_YLD_1"],
        ["A_YLD_1"],
        ["ISMA_B_YLD", "ISMA_A_YLD"],
        ["OPEN_YLD"],
        None,
    ],
    "index": [["TRDPRC_1"], ["MID_PRICE"], ["BID", "ASK"], ["BID"], ["ASK"], None],
    "fx": [["MID_PRICE"], ["BID", "ASK"], ["TRDPRC_1"], ["BID"], ["ASK"], None],
    "futures_price": [["TRDPRC_1"], ["MID_PRICE"], ["SETTLE"], ["BID", "ASK"], None],
    "commodity": [["TRDPRC_1"], ["MID_PRICE"], ["SETTLE"], ["BID", "ASK"], None],
    "fixing_fx": [["TRDPRC_1"], ["MID_PRICE"], ["VALUE_TS1"], ["BID", "ASK"], ["BID"], ["ASK"], None],
}

YIELD_FIELD_HINTS = ("YLD", "YIELD", "YLDTOMAT")
PRICE_FIELD_HINTS = ("TRDPRC", "MID_PRICE", "SETTLE", "BID", "ASK", "VALUE")

def to_lseg_time(dt: datetime) -> str:
    return dt.astimezone(REPORT_TZ).isoformat(timespec="seconds")

def ensure_dt_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = pd.to_datetime(out.index, errors="coerce")
    out = out[~out.index.isna()]
    if getattr(out.index, "tz", None) is None:
        out.index = out.index.tz_localize(REPORT_TZ, nonexistent="shift_forward", ambiguous="NaT")
    else:
        out.index = out.index.tz_convert(REPORT_TZ)
    return out.sort_index()

def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = ["|".join([str(x) for x in tup if str(x) != ""]) for tup in out.columns]
    else:
        out.columns = [str(c) for c in out.columns]
    return out

def _series_from_history(df: pd.DataFrame, requested_fields, unit: str) -> tuple[pd.Series, str]:
    if df is None or len(df) == 0:
        return pd.Series(dtype=float), ""

    df = ensure_dt_index(_flatten_columns(df))
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(how="all")
    if df.empty:
        return pd.Series(dtype=float), ""

    upper_cols = {c.upper(): c for c in df.columns}
    for bid_name, ask_name in [("BID", "ASK"), ("B_YLD_1", "A_YLD_1"), ("ISMA_B_YLD", "ISMA_A_YLD")]:
        b = upper_cols.get(bid_name)
        a = upper_cols.get(ask_name)
        if b and a:
            s = df[[b, a]].mean(axis=1)
            return s.dropna(), f"mid({b},{a})"

    if requested_fields:
        for f in requested_fields:
            for c in df.columns:
                if f.upper() in c.upper():
                    return df[c].dropna(), c

    hints = YIELD_FIELD_HINTS if unit == "yield_pct" else PRICE_FIELD_HINTS
    for h in hints:
        for c in df.columns:
            if h in c.upper():
                return df[c].dropna(), c

    c = df.columns[0]
    return df[c].dropna(), c

def looks_valid(s: pd.Series, unit: str) -> tuple[bool, str]:
    if s is None or s.dropna().empty:
        return False, "empty"
    x = s.dropna()
    med = float(x.median())

    if unit == "yield_pct":
        if med > 20:
            return False, f"yield looks like price/index: median={med:.4f}"
        if med < -5:
            return False, f"yield too low: median={med:.4f}"
    if unit == "fixing_fx":
        if not (4 <= med <= 10):
            return False, f"fixing level suspicious: median={med:.4f}"
    if unit == "fx":
        if med <= 0:
            return False, f"fx level non-positive: median={med:.4f}"
    return True, ""

def fetch_one(asset: dict, start: datetime, end: datetime, interval: str):
    log_rows = []
    unit = asset["unit"]
    field_sets = FIELD_SETS_BY_UNIT.get(unit, [None])

    for ric in asset["rics"]:
        for fields in field_sets:
            try:
                kwargs = {
                    "universe": [ric],
                    "start": to_lseg_time(start),
                    "end": to_lseg_time(end),
                    "interval": interval,
                }
                if fields is not None:
                    kwargs["fields"] = fields

                raw = ld.get_history(**kwargs)
                s, field_used = _series_from_history(raw, fields, unit)
                ok, reason = looks_valid(s, unit)

                if ok:
                    return s.rename(asset["name"]), {
                        **asset,
                        "ric": ric,
                        "field": field_used or str(fields),
                        "rows": int(s.notna().sum()),
                        "status": "ok",
                        "error": "",
                    }

                log_rows.append({
                    **asset,
                    "ric": ric,
                    "field": str(fields),
                    "rows": int(s.notna().sum()) if s is not None else 0,
                    "status": "invalid",
                    "error": reason,
                })

            except Exception as e:
                log_rows.append({
                    **asset,
                    "ric": ric,
                    "field": str(fields),
                    "rows": 0,
                    "status": "failed",
                    "error": str(e)[:300],
                })

    err = log_rows[-1] if log_rows else {**asset, "ric": "", "field": "", "rows": 0, "status": "failed", "error": "no attempt"}
    return None, err

def download_panel(assets: list[dict], start: datetime, end: datetime, interval: str):
    series_list = []
    logs = []
    for asset in sorted(assets, key=lambda x: x["order"]):
        s, log = fetch_one(asset, start, end, interval)
        logs.append(log)
        if s is not None and len(s.dropna()) > 0:
            series_list.append(s)

    panel = pd.concat(series_list, axis=1).sort_index() if series_list else pd.DataFrame()
    log_df = pd.DataFrame(logs).sort_values("order")
    return panel, log_df

In [ ]:
# === 6. 拉取数据 ===

intraday_panel_raw, intraday_log = download_panel(
    wanted_assets("intraday"),
    start=intraday_start,
    end=intraday_end,
    interval=INTRADAY_INTERVAL,
)

daily_panel_raw, daily_log = download_panel(
    wanted_assets("daily"),
    start=daily_start,
    end=daily_end,
    interval=LONG_INTERVAL,
)

display(Markdown("### RIC 拉取日志：5min"))
display(intraday_log[["section", "group", "name", "ric", "field", "rows", "status", "error"]])

display(Markdown("### RIC 拉取日志：daily"))
display(daily_log[["section", "group", "name", "ric", "field", "rows", "status", "error"]])

In [ ]:
# === 7. 衍生指标 ===

def add_derived(panel: pd.DataFrame) -> pd.DataFrame:
    out = panel.copy()

    def add(name, s):
        if s is not None:
            s = s.dropna()
            if len(s) >= 2:
                out[name] = s

    if {"UST 10Y", "UST 2Y"}.issubset(out.columns):
        add("UST 2s10s spread", (out["UST 10Y"] - out["UST 2Y"]) * 100)
    if {"UST 10Y", "UST 5Y"}.issubset(out.columns):
        add("UST 5s10s spread", (out["UST 10Y"] - out["UST 5Y"]) * 100)
    if {"UST 30Y", "UST 5Y"}.issubset(out.columns):
        add("UST 5s30s spread", (out["UST 30Y"] - out["UST 5Y"]) * 100)

    if {"UST 5Y", "TIPS real 5Y"}.issubset(out.columns):
        add("BEI 5Y", out["UST 5Y"] - out["TIPS real 5Y"])
    if {"UST 10Y", "TIPS real 10Y"}.issubset(out.columns):
        add("BEI 10Y", out["UST 10Y"] - out["TIPS real 10Y"])
    if {"UST 30Y", "TIPS real 30Y"}.issubset(out.columns):
        add("BEI 30Y", out["UST 30Y"] - out["TIPS real 30Y"])

    if {"USDCNH", "USDCNY"}.issubset(out.columns):
        add("CNH-CNY spread", out["USDCNH"] - out["USDCNY"])

    if {"USDCNY", "USD/CNY fixing"}.issubset(out.columns):
        add("USDCNY - fixing", out["USDCNY"] - out["USD/CNY fixing"])
    if {"USDCNH", "USD/CNY fixing"}.issubset(out.columns):
        add("USDCNH - fixing", out["USDCNH"] - out["USD/CNY fixing"])

    return out

intraday_panel = add_derived(intraday_panel_raw)
daily_panel = add_derived(daily_panel_raw)

def slice_window(panel: pd.DataFrame, start: datetime, end: datetime) -> pd.DataFrame:
    if panel.empty:
        return panel.copy()
    return panel.loc[(panel.index >= start.astimezone(REPORT_TZ)) & (panel.index <= end.astimezone(REPORT_TZ))].copy()

main_panel = slice_window(intraday_panel, main_start, main_end)
ny_panel = slice_window(intraday_panel, ny_start, ny_end)
rolling24_panel = slice_window(intraday_panel, rolling24_start, rolling24_end)

print("main_panel:", main_panel.shape)
print("ny_panel:", ny_panel.shape)
print("rolling24_panel:", rolling24_panel.shape)
print("daily_panel:", daily_panel.shape)

In [ ]:
# === 8. 摘要表与数据质量 ===

def get_unit(asset: str) -> str:
    return UNIT_BY_NAME.get(asset, "index")

def get_group(asset: str) -> str:
    return GROUP_BY_NAME.get(asset, "Other")

def get_section(asset: str) -> str:
    return SECTION_BY_NAME.get(asset, "Other")

def get_order(asset: str) -> int:
    return ORDER_BY_NAME.get(asset, 9999)

def change_display(first: float, last: float, unit: str):
    chg = last - first
    pct = chg / first if pd.notna(first) and first != 0 else np.nan

    if unit in ("yield_pct", "bei_pct"):
        return chg, chg * 100, "bp", np.nan
    if unit == "spread_bp":
        return chg, chg, "bp", np.nan
    if unit in ("fx", "fixing_fx", "fx_spread"):
        return chg, chg * 10000, "pips", pct if unit == "fx" else np.nan
    if unit in ("index", "futures_price", "commodity"):
        return chg, chg, "pts", pct
    return chg, chg, "pts", pct

def format_level(x: float, unit: str) -> str:
    if pd.isna(x):
        return ""
    if unit in ("yield_pct", "bei_pct"):
        return f"{x:.4f}%"
    if unit == "spread_bp":
        return f"{x:.2f}bp"
    if unit in ("fx", "fixing_fx"):
        return f"{x:.4f}"
    if unit == "fx_spread":
        return f"{x*10000:.1f}pips"
    if unit in ("futures_price", "commodity", "index"):
        return f"{x:.4f}"
    return f"{x:.4f}"

def format_change(x: float, unit: str) -> str:
    if pd.isna(x):
        return ""
    sign = "+" if x > 0 else ""
    if unit in ("bp", "pips"):
        return f"{sign}{x:.2f}{unit}"
    if unit == "pts":
        return f"{sign}{x:.4f}"
    return f"{sign}{x:.4f}"

def summarize_panel(panel: pd.DataFrame, window_name: str) -> pd.DataFrame:
    rows = []
    if panel.empty:
        return pd.DataFrame()

    for col in sorted(panel.columns, key=get_order):
        s = panel[col].dropna()
        if len(s) < 2:
            continue
        unit = get_unit(col)
        first, last = float(s.iloc[0]), float(s.iloc[-1])
        raw_chg, chg_disp, chg_unit, pct = change_display(first, last, unit)

        rows.append({
            "Window": window_name,
            "Section": get_section(col),
            "Group": get_group(col),
            "Asset": col,
            "First Time": s.index[0].strftime("%Y-%m-%d %H:%M"),
            "Last Time": s.index[-1].strftime("%Y-%m-%d %H:%M"),
            "First": first,
            "Last": last,
            "Level": format_level(last, unit),
            "Change": raw_chg,
            "Change Display": chg_disp,
            "Change Text": format_change(chg_disp, chg_unit),
            "Change Unit": chg_unit,
            "% Change": pct,
            "% Change Text": "" if pd.isna(pct) else f"{pct:+.2%}",
            "High": float(s.max()),
            "Low": float(s.min()),
            "Obs": int(s.notna().sum()),
            "Order": get_order(col),
        })

    return pd.DataFrame(rows).sort_values(["Order"]).reset_index(drop=True)

summary_main = summarize_panel(main_panel, "深圳早会主窗口")
summary_ny = summarize_panel(ny_panel, "纽约交易时段")
summary_24h = summarize_panel(rolling24_panel, "滚动24小时")

def rows_by_section(summary: pd.DataFrame, sections: list[str]) -> pd.DataFrame:
    if summary.empty:
        return summary
    return summary[summary["Section"].isin(sections)].copy()

def display_summary(summary: pd.DataFrame, title: str):
    cols = ["Section", "Group", "Asset", "Level", "Change Text", "% Change Text", "High", "Low", "Obs"]
    display(Markdown(f"### {title}"))
    display(summary[cols] if not summary.empty else pd.DataFrame(columns=cols))

display_summary(rows_by_section(summary_main, ["A. Rates", "B. Real & Inflation"]), "主窗口：利率、曲线、真实利率、BEI")
display_summary(rows_by_section(summary_main, ["D. USD & FX", "E. RMB"]), "主窗口：美元、外汇、人民币")
display_summary(rows_by_section(summary_main, ["C. Treasury Futures", "F. Commodities"]), "主窗口：美债期货、商品")

def latest_fixing_info(daily_panel: pd.DataFrame) -> dict:
    if "USD/CNY fixing" not in daily_panel.columns:
        return {"status": "missing", "detail": "USD/CNY fixing not available"}
    s = daily_panel["USD/CNY fixing"].dropna()
    if s.empty:
        return {"status": "missing", "detail": "USD/CNY fixing empty"}
    last_date = s.index[-1].date()
    prev = s.iloc[-2] if len(s) >= 2 else np.nan
    last = s.iloc[-1]
    chg_pips = (last - prev) * 10000 if pd.notna(prev) else np.nan
    stale = last_date != target_fixing_date
    return {
        "status": "stale" if stale else "ok",
        "last_date": last_date,
        "target_date": target_fixing_date,
        "last": float(last),
        "chg_pips": float(chg_pips) if pd.notna(chg_pips) else np.nan,
        "detail": f"latest={last_date}, target={target_fixing_date}, level={last:.4f}",
    }

def data_quality_checks() -> pd.DataFrame:
    issues = []

    for _, r in pd.concat([intraday_log.assign(freq="5min"), daily_log.assign(freq="daily")], ignore_index=True).iterrows():
        if r["status"] != "ok":
            issues.append({
                "Severity": "HIGH" if r["unit"] in ("yield_pct", "fx", "fixing_fx") else "MEDIUM",
                "Asset": r["name"],
                "Issue": f"{r['freq']} 拉取失败/无效",
                "Detail": str(r["error"])[:180],
            })

    for col in ["UST 2Y", "UST 5Y", "UST 10Y", "UST 30Y", "TIPS real 5Y", "TIPS real 10Y", "TIPS real 30Y"]:
        if col in main_panel.columns:
            med = main_panel[col].dropna().median()
            if pd.notna(med) and med > 20:
                issues.append({"Severity": "HIGH", "Asset": col, "Issue": "收益率看起来像价格", "Detail": f"median={med:.4f}"})

    for col in ["BEI 5Y", "BEI 10Y", "BEI 30Y"]:
        if col in main_panel.columns:
            med = main_panel[col].dropna().median()
            if pd.notna(med) and (med < -1 or med > 6):
                issues.append({"Severity": "MEDIUM", "Asset": col, "Issue": "BEI水平异常", "Detail": f"median={med:.4f}%"})

    if "CNH-CNY spread" in main_panel.columns:
        obs = int(main_panel["CNH-CNY spread"].notna().sum())
        if obs < 80:
            issues.append({
                "Severity": "MEDIUM",
                "Asset": "CNH-CNY spread",
                "Issue": "CNY/CNH可比样本偏少",
                "Detail": f"obs={obs}; CNY在岸交易时段较短，spread只在共同时间戳比较。",
            })

    fix = latest_fixing_info(daily_panel)
    if fix["status"] == "missing":
        issues.append({"Severity": "MEDIUM", "Asset": "USD/CNY fixing", "Issue": "中间价缺失", "Detail": fix["detail"]})
    elif fix["status"] == "stale":
        issues.append({
            "Severity": "LOW",
            "Asset": "USD/CNY fixing",
            "Issue": "中间价日期非目标日期",
            "Detail": fix["detail"] + "；9:15前运行通常显示上一交易日中间价。",
        })

    return pd.DataFrame(issues) if issues else pd.DataFrame([{
        "Severity": "OK",
        "Asset": "ALL",
        "Issue": "No major issue",
        "Detail": "核心字段口径和数据完整性未发现重大异常。",
    }])

quality_df = data_quality_checks()
display(Markdown("### 数据质量 / 口径检查"))
display(quality_df)

In [ ]:
# === 9. 早会文字结论 ===

def lookup(summary: pd.DataFrame, asset: str):
    if summary.empty or asset not in set(summary["Asset"]):
        return None
    return summary.loc[summary["Asset"].eq(asset)].iloc[0]

def chg(summary: pd.DataFrame, asset: str) -> str:
    r = lookup(summary, asset)
    if r is None:
        return "n/a"
    return f"{r['Level']}（{r['Change Text']}）"

def build_morning_notes(summary: pd.DataFrame) -> list[str]:
    notes = []

    r2 = lookup(summary, "UST 2Y")
    r10 = lookup(summary, "UST 10Y")
    if r2 is not None and r10 is not None:
        notes.append(
            f"美债：2Y {chg(summary, 'UST 2Y')}，10Y {chg(summary, 'UST 10Y')}；"
            f"曲线2s10s {chg(summary, 'UST 2s10s spread')}。"
        )

    beis = [x for x in ["BEI 5Y", "BEI 10Y", "BEI 30Y"] if lookup(summary, x) is not None]
    if beis:
        notes.append(
            "通胀预期："
            + "，".join([f"{x.replace('BEI ', '')} {chg(summary, x)}" for x in beis])
            + "；BEI口径为名义收益率减TIPS真实收益率。"
        )

    if lookup(summary, "DXY") is not None:
        rmb_bits = []
        for x in ["USDCNY", "USDCNH", "CNH-CNY spread"]:
            if lookup(summary, x) is not None:
                rmb_bits.append(f"{x} {chg(summary, x)}")
        notes.append(
            f"美元：DXY {chg(summary, 'DXY')}；"
            + ("人民币：" + "，".join(rmb_bits) + "。" if rmb_bits else "")
        )

    bits = []
    for x in ["TY 10Y Treasury Fut", "Brent crude", "WTI crude", "Gold", "Copper"]:
        if lookup(summary, x) is not None:
            bits.append(f"{x} {chg(summary, x)}")
    if bits:
        notes.append("联动资产：" + "，".join(bits) + "。")

    return notes

morning_notes = build_morning_notes(summary_main)
display(Markdown("### 一屏结论"))
for n in morning_notes:
    display(Markdown(f"- {n}"))

In [ ]:
# === 10. 交易时段与事件日历：保留简洁版 ===

TRADING_HOURS = pd.DataFrame([
    {
        "模块": "美债期货 / WTI / 黄金 / 铜",
        "主要市场": "CME/CBOT/NYMEX/COMEX Globex",
        "美东时间": "Sun-Fri 18:00-17:00，17:00-18:00日度休市",
        "深圳时间": "夏令约06:00-次日05:00；冬令约07:00-次日06:00",
        "早会意义": "9点已覆盖上一纽约交易段，并包含亚洲早盘早段。",
    },
    {
        "模块": "Brent",
        "主要市场": "ICE Futures Europe",
        "美东时间": "约Sun-Fri 20:00-18:00，具体以ICE为准",
        "深圳时间": "夏令约08:00-次日06:00；冬令约09:00-次日07:00",
        "早会意义": "9点附近刚进入/刚过Brent新交易日早段。",
    },
    {
        "模块": "FX / CNH",
        "主要市场": "OTC FX",
        "美东时间": "Sun 17:00-Fri 17:00近24小时",
        "深圳时间": "周一早至周六清晨近24小时",
        "早会意义": "CNH适合观察隔夜美元与亚洲早盘联动。",
    },
    {
        "模块": "CNY即期",
        "主要市场": "CFETS",
        "美东时间": "约前一日21:30-15:00（夏令）",
        "深圳时间": "09:30-次日03:00",
        "早会意义": "9点尚未开盘，主看上一交易日收盘和CNH预期。",
    },
    {
        "模块": "人民币中间价",
        "主要市场": "PBOC/CFETS",
        "美东时间": "约前一日21:15/20:15",
        "深圳时间": "通常09:15左右",
        "早会意义": "9点跑一般是上一交易日中间价；9:20后重跑可看当日。",
    },
])

# 重要数据时间：日期可维护，时间自动换算。
# 建议每月初补一次 FOMC、CPI、PCE、NFP、GDP、Retail Sales 等关键日期。
MANUAL_US_EVENTS = [
    # 示例：
    # {"Event": "FOMC statement", "Time_ET": "2026-06-17 14:00", "Importance": "High"},
    # {"Event": "CPI", "Time_ET": "2026-05-12 08:30", "Importance": "High"},
]

def make_event_calendar(manual_events: list[dict], asof: datetime, horizon_days: int = 14) -> pd.DataFrame:
    rows = []
    for e in manual_events:
        try:
            t_et = pd.Timestamp(e["Time_ET"], tz=NY_TZ)
        except Exception:
            continue
        t_sh = t_et.tz_convert(REPORT_TZ)
        if asof - timedelta(days=1) <= t_sh.to_pydatetime() <= asof + timedelta(days=horizon_days):
            rows.append({
                "Event": e["Event"],
                "Time ET": t_et.strftime("%Y-%m-%d %H:%M"),
                "深圳时间": t_sh.strftime("%Y-%m-%d %H:%M"),
                "Importance": e.get("Importance", ""),
            })

    # 每周初请：周四08:30 ET，规则生成，若遇节假日请手动覆盖。
    start = asof.astimezone(NY_TZ).date() - timedelta(days=1)
    for d in pd.date_range(start, periods=horizon_days + 3, freq="D").date:
        if pd.Timestamp(d).weekday() == 3:
            t_et = datetime.combine(d, dt_time(8, 30), tzinfo=NY_TZ)
            t_sh = t_et.astimezone(REPORT_TZ)
            rows.append({
                "Event": "Initial Jobless Claims",
                "Time ET": t_et.strftime("%Y-%m-%d %H:%M"),
                "深圳时间": t_sh.strftime("%Y-%m-%d %H:%M"),
                "Importance": "Medium",
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return pd.DataFrame(columns=["Event", "Time ET", "深圳时间", "Importance"])
    return out.drop_duplicates().sort_values("深圳时间").reset_index(drop=True)

event_calendar = make_event_calendar(MANUAL_US_EVENTS, asof_dt)
display(Markdown("### 交易时段对照"))
display(TRADING_HOURS)
display(Markdown("### 美国重要数据日历（简洁版）"))
display(event_calendar)

In [ ]:
# === 11. 绘图函数 ===

def existing(panel: pd.DataFrame, cols: list[str]) -> list[str]:
    return [c for c in cols if c in panel.columns and panel[c].dropna().shape[0] >= 2]

def line_fig(panel: pd.DataFrame, cols: list[str], title: str, ytitle: str, normalize: bool = False, height: int = 420):
    cols = existing(panel, cols)
    fig = go.Figure()
    if not cols:
        fig.update_layout(title=f"{title}（无可用数据）", template=plotly_template, height=height)
        return fig

    for col in cols:
        s = panel[col].dropna()
        y = s / s.iloc[0] * 100 if normalize else s
        fig.add_trace(go.Scatter(
            x=s.index.tz_convert(REPORT_TZ),
            y=y,
            mode="lines",
            name=col,
            hovertemplate="%{x}<br>%{y:.4f}<extra>%{fullData.name}</extra>",
        ))

    fig.update_layout(
        title=title,
        template=plotly_template,
        hovermode="x unified",
        height=height,
        margin=dict(l=40, r=20, t=60, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0),
        yaxis_title=ytitle,
        xaxis_title="",
    )
    return fig

def change_bar_fig(summary: pd.DataFrame, sections: list[str], title: str):
    df = rows_by_section(summary, sections)
    if df.empty:
        fig = go.Figure()
        fig.update_layout(title=f"{title}（无可用数据）", template=plotly_template, height=360)
        return fig

    df = df.copy().dropna(subset=["Change Display"])
    fig = go.Figure(go.Bar(
        x=df["Asset"],
        y=df["Change Display"],
        text=df["Change Text"],
        textposition="outside",
        hovertemplate="%{x}<br>%{y:.4f} %{customdata}<extra></extra>",
        customdata=df["Change Unit"],
    ))
    fig.update_layout(
        title=title,
        template=plotly_template,
        height=380,
        margin=dict(l=40, r=20, t=60, b=80),
        yaxis_title="Change",
        xaxis_tickangle=-25,
    )
    fig.add_hline(y=0, line_width=1)
    return fig

def make_figures():
    figs = []
    figs.append(line_fig(main_panel, ["UST 2Y", "UST 5Y", "UST 10Y", "UST 30Y"], "主窗口：美债名义收益率", "Yield (%)"))
    figs.append(change_bar_fig(summary_main, ["A. Rates", "B. Real & Inflation"], "主窗口：利率、曲线、BEI变化"))
    figs.append(line_fig(main_panel, ["TIPS real 5Y", "TIPS real 10Y", "TIPS real 30Y"], "主窗口：TIPS真实收益率", "Real yield (%)"))
    figs.append(line_fig(main_panel, ["BEI 5Y", "BEI 10Y", "BEI 30Y"], "主窗口：盈亏平衡通胀 BEI", "BEI (%)"))
    figs.append(line_fig(main_panel, ["TU 2Y Treasury Fut", "FV 5Y Treasury Fut", "TY 10Y Treasury Fut", "US 30Y Treasury Fut", "WN Ultra Bond Fut"], "主窗口：CBOT美债期货（归一化）", "Start=100", normalize=True))
    figs.append(line_fig(main_panel, ["DXY"], "主窗口：美元指数 DXY", "Index"))
    figs.append(line_fig(main_panel, ["EURUSD", "USDJPY", "GBPUSD", "AUDUSD"], "主窗口：G10主要汇率（归一化）", "Start=100", normalize=True))
    figs.append(line_fig(main_panel, ["USDCNY", "USDCNH"], "主窗口：USDCNY / USDCNH", "Spot"))
    figs.append(line_fig(daily_panel.tail(260), ["USD/CNY fixing", "USDCNY", "USDCNH"], "近一年：人民币中间价 vs CNY/CNH", "USD/CNY"))
    figs.append(line_fig(main_panel, ["Brent crude", "WTI crude", "Gold", "Copper"], "主窗口：油铜金（归一化）", "Start=100", normalize=True))
    figs.append(line_fig(daily_panel.tail(260), ["UST 2Y", "UST 5Y", "UST 10Y", "UST 30Y"], "近一年：美债收益率", "Yield (%)"))
    figs.append(line_fig(daily_panel.tail(260), ["BEI 5Y", "BEI 10Y", "BEI 30Y"], "近一年：BEI", "BEI (%)"))
    return figs

figs = make_figures()
for fig in figs:
    display(fig)

In [ ]:
# === 12. 输出漂亮清晰的 HTML 报告 ===

timestamp = datetime.now(REPORT_TZ).strftime("%Y%m%d_%H%M")
html_path = f"{OUTPUT_DIR}/lseg_ust_usd_shenzhen_morning_dashboard_v6_{timestamp}.html"
csv_path = f"{OUTPUT_DIR}/lseg_ust_usd_summary_v6_{timestamp}.csv"
log_path = f"{OUTPUT_DIR}/lseg_ust_usd_ric_log_v6_{timestamp}.csv"

summary_main.to_csv(csv_path, index=False, encoding="utf-8-sig")
pd.concat([intraday_log.assign(freq="5min"), daily_log.assign(freq="daily")], ignore_index=True).to_csv(log_path, index=False, encoding="utf-8-sig")

def html_table(df: pd.DataFrame, cols: list[str] | None = None, max_rows: int | None = None) -> str:
    if cols:
        df = df[[c for c in cols if c in df.columns]].copy()
    if max_rows:
        df = df.head(max_rows)
    return df.to_html(index=False, escape=False, classes="clean-table", border=0)

summary_cols = ["Group", "Asset", "Level", "Change Text", "% Change Text", "High", "Low", "Obs"]

rates_tbl = html_table(rows_by_section(summary_main, ["A. Rates", "B. Real & Inflation"]), summary_cols)
usd_rmb_tbl = html_table(rows_by_section(summary_main, ["D. USD & FX", "E. RMB"]), summary_cols)
fut_cmd_tbl = html_table(rows_by_section(summary_main, ["C. Treasury Futures", "F. Commodities"]), summary_cols)

fix = latest_fixing_info(daily_panel)
fixing_card = "不可用"
if fix.get("status") in ("ok", "stale"):
    chg_txt = "" if pd.isna(fix.get("chg_pips", np.nan)) else f"{fix['chg_pips']:+.1f} pips"
    fixing_card = f"{fix['last']:.4f}｜{chg_txt}<br><span class='muted'>latest {fix['last_date']} / target {fix['target_date']}</span>"

def metric_card(title, body):
    return f"<div class='card'><div class='card-title'>{title}</div><div class='card-body'>{body}</div></div>"

cards_html = "".join([
    metric_card("As-of", f"{asof_dt.strftime('%Y-%m-%d %H:%M')}<br><span class='muted'>Asia/Shanghai</span>"),
    metric_card("主窗口", f"{main_start.strftime('%m-%d %H:%M')} → {main_end.strftime('%m-%d %H:%M')}<br><span class='muted'>上一中国收盘后至早会</span>"),
    metric_card("10Y UST", chg(summary_main, "UST 10Y")),
    metric_card("DXY", chg(summary_main, "DXY")),
    metric_card("USD/CNY fixing", fixing_card),
])

notes_html = "<ul>" + "".join([f"<li>{n}</li>" for n in morning_notes]) + "</ul>"

fig_html = []
for i, fig in enumerate(figs):
    fig_html.append(fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False))
figs_block = "\n".join(fig_html)

quality_html = html_table(quality_df)
trading_html = html_table(TRADING_HOURS)
events_html = html_table(event_calendar)
log_html = html_table(pd.concat([intraday_log.assign(freq="5min"), daily_log.assign(freq="daily")], ignore_index=True)[
    ["freq", "section", "group", "name", "ric", "field", "rows", "status", "error"]
])

html = f"""
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>LSEG 深圳早会看板 v6</title>
<style>
body {{
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "PingFang SC", "Microsoft YaHei", Arial, sans-serif;
    background: #f6f7fb;
    color: #111827;
    margin: 0;
}}
.container {{
    max-width: 1280px;
    margin: 0 auto;
    padding: 24px;
}}
.header {{
    background: linear-gradient(135deg, #111827, #334155);
    color: white;
    padding: 28px 32px;
    border-radius: 22px;
    box-shadow: 0 12px 30px rgba(15,23,42,.18);
}}
.header h1 {{
    margin: 0 0 8px 0;
    font-size: 30px;
}}
.subtitle {{
    color: #d1d5db;
    font-size: 14px;
    line-height: 1.6;
}}
.cards {{
    display: grid;
    grid-template-columns: repeat(5, minmax(0, 1fr));
    gap: 14px;
    margin: 18px 0 22px 0;
}}
.card {{
    background: white;
    border-radius: 18px;
    padding: 16px;
    box-shadow: 0 8px 20px rgba(15,23,42,.08);
}}
.card-title {{
    font-size: 13px;
    color: #6b7280;
    margin-bottom: 8px;
}}
.card-body {{
    font-size: 18px;
    font-weight: 700;
    line-height: 1.35;
}}
.muted {{
    color: #6b7280;
    font-size: 12px;
    font-weight: 400;
}}
.section {{
    background: white;
    border-radius: 20px;
    padding: 18px 20px;
    margin: 18px 0;
    box-shadow: 0 8px 20px rgba(15,23,42,.06);
}}
.section h2 {{
    font-size: 20px;
    margin: 0 0 12px 0;
    border-left: 5px solid #2563eb;
    padding-left: 10px;
}}
.clean-table {{
    border-collapse: collapse;
    width: 100%;
    font-size: 13px;
}}
.clean-table th {{
    text-align: left;
    background: #f3f4f6;
    color: #374151;
    padding: 9px 10px;
    border-bottom: 1px solid #e5e7eb;
    white-space: nowrap;
}}
.clean-table td {{
    padding: 8px 10px;
    border-bottom: 1px solid #eef2f7;
    white-space: nowrap;
}}
.clean-table tr:hover {{
    background: #f9fafb;
}}
details {{
    margin-top: 10px;
}}
summary {{
    cursor: pointer;
    color: #2563eb;
    font-weight: 600;
}}
ul {{
    margin-top: 8px;
    line-height: 1.75;
}}
@media (max-width: 1000px) {{
    .cards {{ grid-template-columns: repeat(2, minmax(0, 1fr)); }}
}}
</style>
</head>
<body>
<div class="container">
    <div class="header">
        <h1>LSEG 美债收益率 & 美元晨间看板</h1>
        <div class="subtitle">
            深圳早会版 v6｜主窗口：上一中国交易日16:00 → 今日09:00｜
            纽约窗口：{ny_start.strftime('%m-%d %H:%M')} → {ny_end.strftime('%m-%d %H:%M')} Asia/Shanghai｜
            生成时间：{datetime.now(REPORT_TZ).strftime('%Y-%m-%d %H:%M:%S')}
        </div>
    </div>

    <div class="cards">{cards_html}</div>

    <div class="section">
        <h2>一屏结论</h2>
        {notes_html}
    </div>

    <div class="section">
        <h2>1. 利率、曲线、真实利率、BEI</h2>
        {rates_tbl}
    </div>

    <div class="section">
        <h2>2. 美元、外汇、人民币</h2>
        {usd_rmb_tbl}
    </div>

    <div class="section">
        <h2>3. 美债期货、油铜金</h2>
        {fut_cmd_tbl}
    </div>

    <div class="section">
        <h2>4. 图表</h2>
        {figs_block}
    </div>

    <div class="section">
        <h2>5. 交易时段对照</h2>
        {trading_html}
    </div>

    <div class="section">
        <h2>6. 美国重要数据日历</h2>
        {events_html}
        <p class="muted">说明：FOMC、CPI、PCE、NFP、GDP、Retail Sales 等高重要性日期建议填入 MANUAL_US_EVENTS；周度初请按周四 08:30 ET 自动生成。</p>
    </div>

    <div class="section">
        <h2>7. 数据质量 / 口径检查</h2>
        {quality_html}
        <details>
            <summary>展开 RIC 拉取日志</summary>
            {log_html}
        </details>
    </div>
</div>
</body>
</html>
"""

with open(html_path, "w", encoding="utf-8") as f:
    f.write(html)

display(HTML(f"""
<div style="padding:14px;border-radius:14px;background:#eef6ff;border:1px solid #bfdbfe">
<b>输出完成</b><br>
HTML 报告：<a href="{html_path}" target="_blank">{html_path}</a><br>
CSV 摘要：<a href="{csv_path}" target="_blank">{csv_path}</a><br>
RIC 日志：<a href="{log_path}" target="_blank">{log_path}</a>
</div>
"""))
print(html_path)
print(csv_path)
print(log_path)

## 每天早会怎么用

建议固定流程：

1. **9:00 跑**：看主窗口，复盘海外隔夜市场和亚洲早盘开端。
2. **9:20 后可重跑**：若需要当日人民币中间价，9:15 前通常只能看到上一交易日 fixing。
3. 先看“一屏结论”和前三张表，再看图表。
4. 如果数据质量里出现 HIGH，先不要给经理转发，优先看 RIC 日志确认字段口径。
5. 每月初把 FOMC、CPI、PCE、NFP、GDP、Retail Sales 等关键日期补到 `MANUAL_US_EVENTS`。